In [9]:
import os
from google.colab import userdata

MOCK_LLM = os.getenv("MOCK_LLM", "1").strip() != "0"

DOCS_DIR = os.getenv("DOCS_DIR", "./docs")
CHROMA_PERSIST_DIR = os.getenv("CHROMA_PERSIST_DIR", "./chroma_db")
COLLECTION_NAME = "zepto_policies"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

In [5]:
from typing import List, Optional
from pydantic import BaseModel, Field

class AskRequest(BaseModel):
    query: str = Field(..., description="Customer query string")

class AskResponse(BaseModel):
    answer: str = Field(..., description="Answer text")
    sources: List[str] = Field(default_factory=list, description="IDs of source documents used")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Confidence score between 0.0 and 1.0")

In [11]:
import os
import glob
!pip install chromadb
import chromadb
from sentence_transformers import SentenceTransformer
from config import DOCS_DIR, CHROMA_PERSIST_DIR, COLLECTION_NAME, EMBEDDING_MODEL_NAME

def ingest_corpus():
    """Loads all 8 documents, embeds them with all-MiniLM-L6-v2, and persists in ChromaDB."""
    print("Initializing embedding model...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

    # Reset collection if exists to guarantee fresh ingestion
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    doc_paths = sorted(glob.glob(os.path.join(DOCS_DIR, "doc_*.txt")))
    if not doc_paths:
        raise FileNotFoundError(f"No document files found in {DOCS_DIR}")

    documents = []
    ids = []
    metadatas = []

    for path in doc_paths:
        filename = os.path.basename(path)
        doc_id = os.path.splitext(filename)[0]

        with open(path, "r", encoding="utf-8") as f:
            content = f.read().strip()

        documents.append(content)
        ids.append(doc_id)
        metadatas.append({"doc_id": doc_id, "source": filename})

    print(f"Generating embeddings for {len(documents)} documents...")
    embeddings = model.encode(documents).tolist()

    collection.add(
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Successfully ingested {len(documents)} documents into ChromaDB collection '{COLLECTION_NAME}'.")

if __name__ == "__main__":
    ingest_corpus()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 832.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

ModuleNotFoundError: No module named 'config'

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """ROLE:
You are an official Customer Support Assistant for Zepto quick commerce. Your duty is to provide strictly grounded, accurate, and concise support responses based solely on official company policies.

CONTEXT:
Retrieved Policy Documents:
{context}

TASK:
1. Analyze the customer's query against the provided retrieved policy documents.
2. If the answer is present in the context, synthesize a clear response, list the source document IDs used (e.g., ["doc_01"]), and set a confidence score (0.0 to 1.0).
3. If the context does not contain relevant information, state that you cannot answer based on available policy documents, set sources to [], and confidence to 0.0.

NEGATIVE CONSTRAINTS:
- Do NOT answer using information not present in the provided context.
- Do NOT make assumptions, invent policies, or speculate on unmentioned fees or timelines.
- Do NOT output any conversational preamble or markdown outside the required JSON schema.

FORMAT & LENGTH:
Output must be a valid JSON object with keys "answer", "sources", and "confidence".
The answer must be concise (1 to 3 sentences maximum).

FEW-SHOT EXAMPLE:
User Query: "Can I cancel my order after it has been packed?"
Retrieved Context:
[doc_05]: Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app...

Expected Output:
{{
  "answer": "No, once an order status changes to 'Packed', it can no longer be cancelled through the app as a rider is dispatched immediately.",
  "sources": ["doc_05"],
  "confidence": 1.0
}}

CURRENT USER QUERY:
{query}
"""

In [ ]:
import json
import chromadb
from typing import List, TypedDict
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

from config import MOCK_LLM, CHROMA_PERSIST_DIR, COLLECTION_NAME, EMBEDDING_MODEL_NAME, GROQ_API_KEY
from schemas import AskResponse
from prompt_template import SYSTEM_PROMPT_TEMPLATE

# Initialize shared embedding model & ChromaDB client (runs locally for real in both modes)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

# LangGraph State Definition
class GraphState(TypedDict):
    query: str
    intent: str
    retrieved_chunks: List[dict]
    response: AskResponse

# Keywords for Task 3 mock intent classification
POLICY_KEYWORDS = [
    "delivery", "return", "refund", "membership",
    "tracking", "cancel", "gift card", "support hours"
]

def classify_intent(state: GraphState) -> GraphState:
    """Node 1: Classifies query as 'policy_question' or 'general_question'."""
    query = state["query"]

    if MOCK_LLM:
        # Mock mode (graded baseline): keyword heuristic
        lowered = query.lower()
        if any(keyword in lowered for keyword in POLICY_KEYWORDS):
            intent = "policy_question"
        else:
            intent = "general_question"
    else:
        # Optional MOCK_LLM=0 real LLM call
        from langchain_groq import ChatGroq
        llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama-3.1-8b-instant")
        res = llm.invoke(f"Classify query as either 'policy_question' or 'general_question'. Output only the label.\nQuery: {query}")
        intent = "policy_question" if "policy_question" in res.content.lower() else "general_question"

    state["intent"] = intent
    return state

def retrieve_and_answer(state: GraphState) -> GraphState:
    """Node 2: Retrieves top-3 chunks from ChromaDB and answers policy questions."""
    query = state["query"]

    # Retrieval step ALWAYS runs for real in both modes
    query_embedding = embedding_model.encode([query]).tolist()
    collection = chroma_client.get_or_create_collection(COLLECTION_NAME)

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )

    retrieved_chunks = []
    if results and results.get("documents") and results["documents"][0]:
        docs = results["documents"][0]
        ids = results["ids"][0]
        for doc, doc_id in zip(docs, ids):
            retrieved_chunks.append({"id": doc_id, "text": doc})

    state["retrieved_chunks"] = retrieved_chunks

    if MOCK_LLM:
        # Mock mode (graded baseline): canned template from top chunk
        top_snippet = retrieved_chunks[0]["text"][:200] if retrieved_chunks else ""
        answer_text = f"Based on the retrieved context: {top_snippet}"
        sources = [retrieved_chunks[0]["id"]] if retrieved_chunks else []

        state["response"] = AskResponse(
            answer=answer_text,
            sources=sources,
            confidence=1.0
        )
    else:
        # Optional MOCK_LLM=0 real LLM call with retry logic
        from langchain_groq import ChatGroq
        llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama-3.1-8b-instant", temperature=0.0)

        context_str = "\n".join([f"[{c['id']}]: {c['text']}" for c in retrieved_chunks])
        prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context_str, query=query)

        validated_response = None
        attempts = 0
        current_prompt = prompt

        while attempts < 3 and not validated_response:
            try:
                raw_out = llm.invoke(current_prompt).content.strip()
                data = json.loads(raw_out)
                validated_response = AskResponse(**data)
            except Exception as e:
                attempts += 1
                current_prompt = f"{prompt}\n\nERROR: Previous output failed JSON validation: {str(e)}. Output ONLY valid JSON."

        if not validated_response:
            validated_response = AskResponse(
                answer="Error: Failed to produce structured response from LLM after retries.",
                sources=[],
                confidence=0.0
            )
        state["response"] = validated_response

    return state

def direct_answer(state: GraphState) -> GraphState:
    """Node 3: Direct answer for general non-policy questions."""
    if MOCK_LLM:
        # Mock mode (graded baseline)
        state["response"] = AskResponse(
            answer="I can only answer questions about Zepto policies right now.",
            sources=[],
            confidence=1.0
        )
    else:
        # Optional MOCK_LLM=0 real LLM call
        from langchain_groq import ChatGroq
        llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama-3.1-8b-instant")
        res = llm.invoke(f"Respond politely explaining you specialize in Zepto policies.\nUser: {state['query']}")
        state["response"] = AskResponse(
            answer=res.content.strip(),
            sources=[],
            confidence=1.0
        )
    return state

def route_intent(state: GraphState) -> str:
    """Conditional Edge routing based on classified intent."""
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    return "direct_answer"

# Build LangGraph Workflow
workflow = StateGraph(GraphState)
workflow.add_node("classify_intent", classify_intent)
workflow.add_node("retrieve_and_answer", retrieve_and_answer)
workflow.add_node("direct_answer", direct_answer)

workflow.set_entry_point("classify_intent")
workflow.add_conditional_edges(
    "classify_intent",
    route_intent,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)
workflow.add_edge("retrieve_and_answer", END)
workflow.add_edge("direct_answer", END)

app_graph = workflow.compile()

In [ ]:
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from schemas import AskRequest, AskResponse
from ingest import ingest_corpus
from graph import app_graph
from config import MOCK_LLM

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Ensure corpus is embedded into ChromaDB on startup
    ingest_corpus()
    yield

app = FastAPI(
    title="Zepto Support Assistant",
    version="1.0.0",
    lifespan=lifespan
)

@app.get("/")
def health():
    return {"status": "ok", "mock_llm": MOCK_LLM}

@app.post("/ask", response_model=AskResponse)
def ask_question(request: AskRequest):
    try:
        initial_state = {
            "query": request.query,
            "intent": "",
            "retrieved_chunks": [],
            "response": None
        }
        final_state = app_graph.invoke(initial_state)
        return final_state["response"]
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))